# Cube555: Transformer + MLP
Choose a puzzle and beam width, then **Run All**. On Kaggle enable **GPU T4 ×2** and Internet.

The small installation cell is folded away. On a Linux GPU node the same notebook works after GPUs have been allocated. First-time compilation needs CUDA PyTorch, `nvcc` and a C++ compiler.


In [ ]:
# First-time installation (also runs automatically with Run All).
from urllib.request import urlretrieve
from runpy import run_path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as setup_dir:
    setup_file, _ = urlretrieve("https://raw.githubusercontent.com/TryDotAtwo/MultiGPUBeamSearch/fa5eff61277c56fdab20db3609a6a0f657280027/kaggle/cube555_github/setup.py", setup_dir + "/setup.py")
    run_path(setup_file)


In [ ]:
import cayleypy as cp
import multigpubeamsearch as beam
from multigpubeamsearch import cube555


## Puzzle and models

In [ ]:
puzzle_id = 1020
beam_width = 1_048_576
num_gpus = None  # use all visible GPUs; or set a number

puzzle = cube555.load(puzzle_id)
graph = cp.CayleyGraph(puzzle.definition, device="cpu", random_seed=42)
predictor = puzzle.blend(graph, transformer_weight=0.8)  # MLP: 0.2


## Search
GPU backend, buffers and compatible profiles are selected automatically. You can repeat the search cell before closing the session.

In [ ]:
beam.enable_native(beam.NativeOptions(num_gpus=num_gpus, timeout_seconds=12 * 3600,
                                      calibration_pipeline_seconds=1800))
session = beam.NativeSession()


In [ ]:
with session.activate():
    result = graph.beam_search(
        start_state=puzzle.start_state,
        predictor=predictor,
        backend="native",
        beam_width=beam_width,
        max_steps=140,
        beam_mode="simple",
        use_child_scores=True,
        return_path=True,
    )


## Solution

In [ ]:
try:
    saved = cube555.save_result(result, graph, puzzle.start_state)
    if result.path_found:
        print(f"Solved in {len(result.path)} moves:", result.path)
    else:
        print("No solution found within 140 steps.")
    print("Saved:", saved)
finally:
    session.close()
